# 01: Exploratory data analysis

*Educational project, not for clinical use.* Interactive companion to `scripts/inspect_data.py`
(Phase 1). Every number comes from the files on disk; nothing about the class list is hard-coded.

In [ ]:
# Run from the repo root (or adjust). Uses the installed `mds` package.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = Path(os.environ.get("BONESCOPE_DATA", "data/raw"))
print(ROOT, DATA_ROOT, DATA_ROOT.exists())

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from mds.data import index_images, token_position_stats, filename_skeleton
df = index_images(DATA_ROOT)
counts = df["label_name"].value_counts()
print(f"{len(counts)} classes, {len(df):,} images")
counts.to_frame("images")

In [ ]:
ax = counts.sort_values().plot.barh(figsize=(7, 0.3 * len(counts) + 1), log=True)
ax.set_xlabel("images (log scale)"); ax.set_title("Class imbalance"); plt.tight_layout()

## Image sizes
Crop size tracks cell size (box + 10%), so it is itself informative.

In [ ]:
from PIL import Image
sizes = pd.DataFrame([Image.open(DATA_ROOT / p).size for p in df["path"]], columns=["w", "h"])
df = df.join(sizes)
df.groupby("label_name")["w"].describe()[["mean", "50%", "min", "max"]].sort_values("50%")

## Identifier evidence
Do filenames carry patient/slide IDs? (~125 unique values at a token position would be suspicious.)

In [ ]:
from collections import Counter
stems = [Path(p).stem for p in df["path"]]
print(Counter(filename_skeleton(s) for s in stems).most_common(10))
pd.DataFrame([t.__dict__ for t in token_position_stats(stems)])

## Rare classes (< 20 images)
Candidates for merging into lineage 'other' buckets; a team decision.

In [ ]:
counts[counts < 20]